<a href="https://colab.research.google.com/github/enyachang1119-debug/DS2002/blob/main/Copy_of_2026_10_16_%E2%80%94_Walmart_Warm_Up_SKU_Mess_%E2%80%94_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Walmart Warm-Up SKU Mess

**Lab — 2026-10-16 · Fall 2026**  

---

## Lab 08 — Walmart Warm-Up (SKU Mess)

This lab uses the **real midterm data** in `../06-midterm-walmart/data/`. It's a warm-up so you hit the ground running on the project. On Kaggle/Colab, upload the data folder or adjust the path.

*If the file isn't found, the cell falls back to a small inline sample so the notebook still runs.*

In [ ]:
from google.colab import files
uploaded = files.upload()

import pandas as pd

PATH = '/content/product_catalog.csv'
cat = pd.read_csv(PATH)

print("Loaded CSV:", PATH)
print("Total rows:", len(cat))
cat.head()

### Q1 — How many rows look like Pop-Tart products?

Filter on `product_name` containing `pop`, case-insensitively. Print the count and the matching rows.

In [ ]:
# TODO (hint: .str.lower().str.contains('pop'))

# Filter rows where product_name contains "pop" (case-insensitive)
pop_tarts = cat[cat["product_name"].str.lower().str.contains("pop", na=False)]
# Count the number of matching rows
print(len(pop_tarts))
# Display the matching rows
print(pop_tarts)

### Q2 — Which SKUs and names all mean *strawberry* Pop-Tart?

List them. Then note which Pop-Tart rows are **not** strawberry — those must stay separate, and saying so is part of the answer.

In [ ]:
# TODO
# Find Pop-Tart products that contain "strawberry"
strawberry = pop_tarts[pop_tarts["product_name"].str.lower().str.contains("strawberry", na=False)]

# Display the SKUs and product names
print(strawberry[["sku", "product_name"]])

# Find Pop-Tart products that are not strawberry
other_pop_tarts = pop_tarts[~pop_tarts["product_name"].str.lower().str.contains("strawberry", na=False)]
print("Pop-Tart rows that are a different product:")
print(other_pop_tarts[["sku", "product_name"]])

**Pop-Tart rows that are a different product:** _There are 4 Pop-Tart rows in the dataset, and all are strawberry flavored. No other flavors are present, as confirmed by the empty `other_pop_tarts` output. If there were other flavors, they should be kept separate and not mapped to the strawberry SKU._

### Q3 — Build the mapping

A dict from each variant SKU to one canonical SKU. Then apply it in a way that leaves unmapped SKUs as themselves rather than turning them into `NaN` — that single mistake silently deletes most of the catalog.

In [ ]:
# TODO
# Map different Strawberry Pop-Tart SKUs to one canonical SKU
sku_mapping = {
    "SKU#459812": "PT-12",
    "POPTART-STRAW": "PT-12",
    "PT_STRAWBERRY": "PT-12"
}

# Save the number of rows before mapping
original_rows = len(cat)

# Apply the mapping and keep unmapped SKUs unchanged
cat["canonical_sku"] = cat["sku"].map(sku_mapping).fillna(cat["sku"])

# Display the original and canonical SKUs
print(cat[["sku", "product_name", "canonical_sku"]])

### Q4 — Normalize `category`

Strip and title-case, then show the distinct values before and after with counts. Note any pair that is still two labels for one thing — `Breakfast` and `Breakfast Foods` are not the same string but they are the same category.

In [ ]:
# TODO
# Show category values and counts before cleaning
print("Before cleaning:")
print(cat["category"].value_counts())

# Remove extra spaces and convert to title case
cat["category_clean"] = cat["category"].str.strip().str.title()

# Show category values and counts after cleaning
print("After cleaning:")
print(cat["category_clean"].value_counts())

# Note that these two categories refer to the same thing
print("Breakfast and Breakfast Foods refer to the same category.")

### Q5 — Find the fragmentation you have not spotted yet

Hand-scanning a catalog does not scale. **TODO:** build a candidate list programmatically — normalize each product name (lowercase, strip punctuation, sort the words) and group by that, then show any group holding more than one SKU.

This is the workflow from Monday's lecture, and it is how you will actually approach the full catalog on the midterm.

In [ ]:
# TODO
# Normalize product names by removing punctuation and sorting words
def normalize_name(name):
    words = re.sub(r"[^\w\s]", " ", str(name).lower()).split()
    return " ".join(sorted(words))

# Apply normalization to each product name
import re
cat["normalized_name"] = cat["product_name"].apply(normalize_name)

# Group by normalized name and find groups with multiple SKUs
duplicates = cat.groupby("normalized_name").filter(
    lambda x: x["sku"].nunique() > 1
)

# Display the matching groups
print(duplicates[["sku", "product_name", "normalized_name"]])

### Q6 — Prove the mapping is safe

**TODO:** assert that applying your mapping produced no nulls in the canonical column and did not change the number of rows. Consolidation renames; it never adds or removes.

In [ ]:
# TODO
assert cat["canonical_sku"].notna().all() # Check that canonical_sku has no missing values
assert len(cat) == original_rows # Check that the number of rows has not changed
print("Mapping is safe: no missing values and row count unchanged.")


### Write-up

**a)** Name two product families beyond Pop-Tarts that appear fragmented, and how you would group them.

**b)** Where does consolidation stop being a data-cleaning task and become a business decision? Give one example from this catalog where you genuinely are not sure, and say who you would ask.

_a: This situation also happens with LED flashlights and Monopoly board games. I would group FLASH-01 and LED-FLASH together because both have the name "FLASHLIGHT LED." Moreover, I would group BG-MONO, MONOPOLY-01, and GAME-MONO together because their names contain the same words in different orders. But before actually merging them, I would make sure their specifications and package sizes are the same._

_b: When product names are similar, but differences in specifications or packaging might affect sales or inventory, merging them requires a business decision. For example, in the dataset, PT-12 is a 4-pack, while POPTART-STRAW is a 6-pack. Even though both are strawberry Pop-Tarts, I am not sure whether they should share a canonical SKU or remain separate based on pack size. I would ask the product manager or inventory team before deciding._